## Cleaning of Aegean Region Rental Housing Data

In [3]:
import os
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('hepsiemlak/01.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   listingCard__media-link href  1560 non-null   str  
 1   listingCard__photo-index      1557 non-null   str  
 2   listingCard__price            1560 non-null   str  
 3   listingCard__date             1560 non-null   str  
 4   listingCard__title            1560 non-null   str  
 5   listingCard__spec-item        1560 non-null   str  
 6   listingCard__spec-item 2      1560 non-null   str  
 7   listingCard__spec-item 3      1560 non-null   str  
 8   listingCard__spec-item 4      1560 non-null   str  
 9   listingCard__spec-sep 4       1538 non-null   str  
 10  listingCard__spec-label 5     1538 non-null   str  
 11  listingCard__spec-item 5      1469 non-null   str  
 12  listingCard__location         1560 non-null   str  
 13  he-lazy-image src             1464 non-null 

['01.csv', '02.csv', '03.csv']

In [8]:
files = os.listdir('hepsiemlak')

df = pd.concat([pd.read_csv(f"hepsiemlak\\{file}") for file in files], ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7441 entries, 0 to 7440
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   listingCard__media-link href  7441 non-null   str   
 1   listingCard__photo-index      7433 non-null   str   
 2   listingCard__price            7441 non-null   object
 3   listingCard__date             7441 non-null   str   
 4   listingCard__title            7441 non-null   str   
 5   listingCard__spec-item        7441 non-null   str   
 6   listingCard__spec-item 2      7441 non-null   str   
 7   listingCard__spec-item 3      7441 non-null   str   
 8   listingCard__spec-item 4      7441 non-null   str   
 9   listingCard__spec-sep 4       5568 non-null   str   
 10  listingCard__spec-label 5     5568 non-null   str   
 11  listingCard__spec-item 5      6880 non-null   str   
 12  listingCard__location         7441 non-null   str   
 13  he-lazy-image src            

In [115]:
try: 
    df = df.drop(columns=[
        "listingCard__media-link href", 
        "listingCard__photo-index", 
        "listingCard__date", 
        "listingCard__title", 
        "listingCard__spec-item", 
        "listingCard__spec-sep 4", 
        "listingCard__spec-label 5", 
        "wp-btn-label", 
        "he-lazy-image src 2",
        "listingCard__owner-name",
        "he-lazy-image src 3"
    ], errors='ignore')

    column_mapping = {
        "listingCard__price": "Price",
        "listingCard__spec-item 2": "Room_Count",
        "listingCard__spec-item 3": "Area_m2",
        "listingCard__spec-item 4": "Building_Age",
        "listingCard__spec-item 5": "Floor",
        "listingCard__location": "Location",
    }

    df[['City', 'District', 'Neighborhood']] = df['Location'].str.split(' / ', expand=True)

    df = df.drop(columns=['Location'])

except Exception as e:
    print(f"Error: {e}")

df = df.rename(columns=column_mapping)

df.head(10)

Error: 'Location'


,Price,Room_Count,Area_m2,Building_Age,Floor,City,District,Neighborhood
0,54.000,2 + 1,80,10,3,İzmir,Karşıyaka,Yalı Mah.
1,23.000,2 + 0,116,3,3,İzmir,Torbalı,Pancar Mah.
2,60.000,3 + 1,140,0,5,İzmir,Karşıyaka,Goncalar Mah.
3,25.000,2 + 1,65,6,3,İzmir,Buca,Atatürk Mah.
4,30.000,3 + 1,135,30,3,İzmir,Buca,Vali Rahmi Bey Mah.
5,42.000,3 + 1,130,30,NaN,İzmir,Çiğli,Evka - 2 Mah.
6,50.000,5 + 1,185,30,NaN,İzmir,Çiğli,Evka - 2 Mah.
7,38.000,3 + 1,115,0,3,İzmir,Karabağlar,Bahar Mah.
8,40.000,2 + 1,100,30,3,İzmir,Karabağlar,Basın Sitesi Mah.
9,45.000,2 + 1,80,46,1,İzmir,Karabağlar,Esenyalı Mah.


In [ ]:
df["Building_Age"]  = df["Building_Age"].str.replace(" Yaşında", "").astype(int)
df["Building_Age"]  = df["Building_Age"].str.replace("Sıfır Bina", "0").astype(int)

In [144]:
df["Area_m2"] = df["Area_m2"].str.replace(" m²", "")
df["Area_m2"] = df["Area_m2"].str.replace(".", "").astype(int)

In [134]:
df["Room_Count"] = df["Room_Count"].apply(lambda x: x.replace("Stüdyo", "1 + 0"))
df["Room"] = df["Room_Count"].apply(lambda x: x.split("+")[0]).astype(int)
df["Living_Room"] = df["Room_Count"].apply(lambda x: x.split("+")[1]).astype(int)

df = df.drop("Room_Count", axis = 1)


In [168]:
floor_mapping = {
    ". Kat": "",
    "Kot ": "-",
    "Giriş Katı": "0",
    "Bodrum ve Zemin": "0",
    "Yarı Bodrum": "0",
    "Bahçe Katı": "0",
    "Zemin": "0",
    "Bodrum": "0",
    "En Üst Kat": "5",
    "Teras Katı": "5",
    "Çatı Katı": "5",
    "Villa Katı": "0",
    "Tripleks": "0",
    "Yüksek Giriş": "1",
    "21 ve üzeri": "21",
    "Ara Kat": "3"
}

df["Floor"] = df["Floor"].fillna("2").astype(str).str.replace("nan", "2", regex=False)

for old_val, new_val in floor_mapping.items():
    df["Floor"] = df["Floor"].str.replace(old_val, new_val, regex=False)

df["Floor"] = pd.to_numeric(df["Floor"].str.strip(), errors='coerce').fillna(2).astype(int)

df.head()

,Price,Area_m2,Building_Age,Floor,City,District,Neighborhood,Room,Living_Room
0,54.000,80,10,3,İzmir,Karşıyaka,Yalı Mah.,2,1
1,23.000,116,3,3,İzmir,Torbalı,Pancar Mah.,2,0
2,60.000,140,0,5,İzmir,Karşıyaka,Goncalar Mah.,3,1
3,25.000,65,6,3,İzmir,Buca,Atatürk Mah.,2,1
4,30.000,135,30,3,İzmir,Buca,Vali Rahmi Bey Mah.,3,1


In [181]:
df["Price"] = df["Price"].astype(str).str.replace(".", "", regex=False).astype(int)

In [182]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7441 entries, 0 to 7440
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Price         7441 non-null   int64
 1   Area_m2       7441 non-null   int64
 2   Building_Age  7441 non-null   int64
 3   Floor         7441 non-null   int64
 4   City          7441 non-null   str  
 5   District      7441 non-null   str  
 6   Neighborhood  7441 non-null   str  
 7   Room          7441 non-null   int64
 8   Living_Room   7441 non-null   int64
dtypes: int64(6), str(3)
memory usage: 523.3 KB


In [183]:
df

,Price,Area_m2,Building_Age,Floor,City,District,Neighborhood,Room,Living_Room
0,54000,80,10,3,İzmir,Karşıyaka,Yalı Mah.,2,1
1,23000,116,3,3,İzmir,Torbalı,Pancar Mah.,2,0
2,60000,140,0,5,İzmir,Karşıyaka,Goncalar Mah.,3,1
3,25000,65,6,3,İzmir,Buca,Atatürk Mah.,2,1
4,30000,135,30,3,İzmir,Buca,Vali Rahmi Bey Mah.,3,1
...,...,...,...,...,...,...,...,...,...
7436,185,79,0,1,Kütahya,Merkez,Evliya Çelebi Mah.,1,1
7437,1475,50,6,3,Kütahya,Merkez,Servi Mah.,1,1
7438,105,50,11,1,Kütahya,Merkez,Fuatpaşa Mah.,1,1
7439,120,50,6,2,Kütahya,Merkez,Balıklı Mah.,1,1


In [ ]:
df.to_csv("data.csv", index=False)